# Degree Choice Assistant Pipeline

Этот notebook запускает весь pipeline по шагам:
1. Past Agent
2. Preferences Agent
3. Market Retrieval
4. Market Agent
5. Final Agent


In [1]:
from __future__ import annotations

import json
from typing import Any

from past_agent import build_past_profile, analyze_past_profile
from preferences_agent import build_preferences_profile, analyze_preferences_profile
from market_retrieval import retrieve_market_context
from market_agent import build_market_profile, analyze_market_profile
from final_agent import analyze_final_recommendation


## 1. Входные данные

Здесь можно менять профиль пользователя и предпочтения.


In [2]:
country = "Israel"

past_inputs = {
    "has_high_school_graduation": "yes",
    "wants_to_continue_same_field": "yes",
    "main_school_focus": "humanities",
    "advanced_subjects": ["management", "finance_economics"],
    "favorite_subjects": ["finance_economics", "literature", "law"],
    "best_subjects": ["finance_economics", "literature"],
}

preferences_inputs = {
    "job_style_preference": "thinking",
    "work_environment_preference": "mixed",
    "preferred_field": "humanities",
    "self_learning_comfort": "no",
    "long_learning_willingness": "yes",
    "learning_structure_preference": "structured",
    "openness_to_new_fields": "no",
}

model = None  # при желании подставь имя модели


## 2. Past Agent


In [3]:
past_profile = build_past_profile(
    country=country,
    has_high_school_graduation=past_inputs["has_high_school_graduation"],
    wants_to_continue_same_field=past_inputs["wants_to_continue_same_field"],
    main_school_focus=past_inputs["main_school_focus"],
    advanced_subjects=past_inputs["advanced_subjects"],
    favorite_subjects=past_inputs["favorite_subjects"],
    best_subjects=past_inputs["best_subjects"],
)

past_result = analyze_past_profile(
    past_profile,
    model=model,
)

print(json.dumps(past_result, indent=2, ensure_ascii=False))


{
  "agent_name": "past_agent",
  "can_recommend": true,
  "degree_or_course": "degree",
  "recommended_fields": [
    "finance_economics",
    "law",
    "literature"
  ],
  "not_recommended_fields": [],
  "uncertain_fields": [],
  "confidence": "high",
  "short_reason": "Strong humanities and social sciences orientation with clear preferences."
}


## 3. Preferences Agent


In [4]:
preferences_profile = build_preferences_profile(
    job_style_preference=preferences_inputs["job_style_preference"],
    work_environment_preference=preferences_inputs["work_environment_preference"],
    preferred_field=preferences_inputs["preferred_field"],
    self_learning_comfort=preferences_inputs["self_learning_comfort"],
    long_learning_willingness=preferences_inputs["long_learning_willingness"],
    learning_structure_preference=preferences_inputs["learning_structure_preference"],
    openness_to_new_fields=preferences_inputs["openness_to_new_fields"],
)

preferences_result = analyze_preferences_profile(
    preferences_profile,
    model=model,
)

print(json.dumps(preferences_result, indent=2, ensure_ascii=False))


{
  "agent_name": "preferences_agent",
  "job_style_preference": "thinking",
  "work_environment_preference": "mixed",
  "preferred_field": "humanities",
  "self_learning_comfort": "no",
  "long_learning_willingness": "yes",
  "learning_structure_preference": "structured",
  "openness_to_new_fields": "no",
  "confidence": "high",
  "short_reason": "The user prefers thinking jobs, a mix of team and alone work, structured learning, and is willing to learn long-term but not open to new fields."
}


## 4. Market Retrieval


In [5]:
market_retrieval_result = retrieve_market_context(
    country=country,
    past_analysis=past_result,
    preferences_analysis=preferences_result,
)

print("Queries:")
for q in market_retrieval_result["queries"]:
    print("-", q)

print("\nResults count:", market_retrieval_result["results_count"])
print("\nMarket context:\n")
print(market_retrieval_result["market_context"])


Queries:
- entry level jobs in demand Israel degree required
- legal assistant jobs Israel degree required
- compliance support jobs Israel degree required
- entry level legal jobs Israel
- career transition from legal to humanities jobs Israel
- career transition from legal to communications jobs Israel
- entry level education writing legal support jobs Israel

Results count: 28

Market context:

Original query: entry level jobs in demand Israel degree required
Title: Nefesh B'Nefesh Israel Job Board
Snippet: Search for potential jobs in various industries throughout Israel, and submit your resume to employers looking for qualified Olim. Start building your career in Israel today!
Source: https://www.nbn.org.il/jobboard/

Original query: entry level jobs in demand Israel degree required
Title: 50 entry level Jobs in Israel, September 2025 | Glassdoor
Snippet: Search Entry level jobs in Israel with company ratings & salaries. 50 open jobs for Entry level in Israel.
Source: https://www.

## 5. Market Agent


In [6]:
market_profile = build_market_profile(
    country=country,
    past_analysis=past_result,
    preferences_analysis=preferences_result,
    market_context=market_retrieval_result["market_context"],
)

market_result = analyze_market_profile(
    market_profile,
    model=model,
)

print(json.dumps(market_result, indent=2, ensure_ascii=False))


{
  "agent_name": "market_agent",
  "can_recommend": true,
  "recommended_paths": [
    {
      "field": "law",
      "career_outcomes": [
        "legal assistant",
        "compliance support",
        "legal secretary"
      ],
      "reason": "Strong market demand for legal support roles in Israel with degree requirements and structured career paths."
    },
    {
      "field": "finance_economics",
      "career_outcomes": [
        "comptroller assistant",
        "financial analyst junior",
        "compliance associate"
      ],
      "reason": "Finance roles are in demand, especially entry-level positions requiring degrees, with clear pathways in Israel's job market."
    },
    {
      "field": "education",
      "career_outcomes": [
        "teaching assistant",
        "entry-level educator",
        "education coordinator"
      ],
      "reason": "Education is a realistic and structured field with demand for degree holders, fitting the user's preference for structured lea

## 6. Final Agent


In [7]:
final_result = analyze_final_recommendation(
    past_result=past_result,
    preferences_result=preferences_result,
    market_result=market_result,
)

print(json.dumps(final_result, indent=2, ensure_ascii=False))


{
  "agent_name": "final_agent",
  "can_recommend": true,
  "user_result": {
    "top_3_recommendations": [
      {
        "field": "law",
        "label": "recommended",
        "confidence": "high",
        "reason": "Strong overall balance between personal fit and market prospects. Possible entry paths include legal assistant, compliance support."
      },
      {
        "field": "literature",
        "label": "recommended",
        "confidence": "high",
        "reason": "Strong overall balance between personal fit and market prospects. Possible entry paths include content writer, editor assistant."
      },
      {
        "field": "finance_economics",
        "label": "recommended",
        "confidence": "medium",
        "reason": "Strong overall balance between personal fit and market prospects. Possible entry paths include comptroller assistant, financial analyst junior."
      }
    ],
    "summary": "The best options are the ones that balance personal fit with realistic fu

## 7. Короткий readable summary


In [8]:
top_3 = final_result["user_result"]["top_3_recommendations"]

print("FINAL RECOMMENDATION")
print("=" * 40)
print(final_result["user_result"]["summary"])
print()

if not top_3:
    print("No top recommendations available.")
    print(final_result["user_result"]["warning"])
else:
    for i, item in enumerate(top_3, start=1):
        print(f"{i}. {item['field']}")
        print(f"   label      : {item['label']}")
        print(f"   confidence : {item['confidence']}")
        print(f"   reason     : {item['reason']}")
        print()

print("TOP CONFLICTS")
print("=" * 40)
for item in final_result["debug_result"].get("top_conflicts", []):
    print(f"- {item['field']} | {item['conflict_type']}")
    print(f"  {item['explanation']}")


FINAL RECOMMENDATION
The best options are the ones that balance personal fit with realistic future opportunities.

1. law
   label      : recommended
   confidence : high
   reason     : Strong overall balance between personal fit and market prospects. Possible entry paths include legal assistant, compliance support.

2. literature
   label      : recommended
   confidence : high
   reason     : Strong overall balance between personal fit and market prospects. Possible entry paths include content writer, editor assistant.

3. finance_economics
   label      : recommended
   confidence : medium
   reason     : Strong overall balance between personal fit and market prospects. Possible entry paths include comptroller assistant, financial analyst junior.

TOP CONFLICTS
- finance_economics | preferences_vs_past
  Supported by both past profile and market evidence.
- education | market_vs_fit
  Education is a realistic and structured field with demand for degree holders, fitting the user's p

## 8. Полный pipeline result


In [9]:
pipeline_result = {
    "country": country,
    "past_profile": past_profile,
    "preferences_profile": preferences_profile,
    "market_retrieval": market_retrieval_result,
    "past_result": past_result,
    "preferences_result": preferences_result,
    "market_profile": market_profile,
    "market_result": market_result,
    "final_result": final_result,
}

print(json.dumps(pipeline_result, indent=2, ensure_ascii=False))


{
  "country": "Israel",
  "past_profile": {
    "country": "Israel",
    "has_high_school_graduation": "yes",
    "wants_to_continue_same_field": "yes",
    "main_school_focus": "humanities",
    "advanced_subjects": [
      "management",
      "finance_economics"
    ],
    "favorite_subjects": [
      "finance_economics",
      "literature",
      "law"
    ],
    "best_subjects": [
      "finance_economics",
      "literature"
    ]
  },
  "preferences_profile": {
    "job_style_preference": "thinking",
    "work_environment_preference": "mixed",
    "preferred_field": "humanities",
    "self_learning_comfort": "no",
    "long_learning_willingness": "yes",
    "learning_structure_preference": "structured",
    "openness_to_new_fields": "no"
  },
  "market_retrieval": {
    "country": "Israel",
    "queries": [
      "entry level jobs in demand Israel degree required",
      "legal assistant jobs Israel degree required",
      "compliance support jobs Israel degree required",
      